## Can gpt2_small store facts already or do I need to work with a different model?

- Load weights into gpt2_model and run prompts that test for evidence of stored facts
- Assess its ability to store facts
- Look online to understand the capabilities of gpt2_small
- Look online for similar assessments

In [6]:
# Setup
import torch
import torch.nn as nn
import einops
from fancy_einsum import einsum
import tqdm.auto as tqdm
import plotly.express as px

from jaxtyping import Float
from functools import partial

# import transformer_lens
import transformer_lens.utilities as utils
from transformer_lens.hook_points import HookPoint

# Hooking utilities
from transformer_lens import FactoredMatrix, HookedTransformer
from transformer_lens.model_bridge import TransformerBridge

In [7]:
def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        labels={"x": xaxis, "y": yaxis},
        **kwargs,
    ).show(renderer)


def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.line(utils.to_numpy(tensor), labels={"x": xaxis, "y": yaxis}, **kwargs).show(renderer)


def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
    x = utils.to_numpy(x)
    y = utils.to_numpy(y)
    px.scatter(y=y, x=x, labels={"x": xaxis, "y": yaxis, "color": caxis}, **kwargs).show(renderer)

In [10]:
t.set_grad_enabled(False)
device = t.device("mps")

gpt2_small = TransformerBridge.boot_transformers("gpt2", device=device)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [23]:
def complete(model, prompt):
    logits, cache = model.run_with_cache(prompt)
    seq = logits.argmax(-1).squeeze(0)
    return model.to_string(seq)

print(complete(gpt2_small, "The president of the United States is: "))


 first of the United States, a
 


Clearly it's going to be difficult for GPT2 to store a fact if it can't complete a sentence like this.

Let's try a simpler example and see what happens

In [24]:
print(complete(gpt2_small, "The colour of the sky is: "))


 first of the sky is a
 


OK we can stop here and upgrade the model. 